# Phase 8 — MLOps & Deployment

**Theory:** Model versioning, experiment tracking, serving models as APIs, containerization.

**Tools covered:** MLflow, FastAPI, Streamlit, Docker (concepts).

**Install:**
```
pip install mlflow fastapi uvicorn streamlit scikit-learn
```

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

import joblib, os, json

np.random.seed(42)
sns.set_theme(style="whitegrid")

# Dataset
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=6, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Data ready: {X_train.shape[0]} train, {X_test.shape[0]} test")

---
## 1. MLflow — Experiment Tracking

MLflow logs parameters, metrics, and models for every experiment run. You can compare runs and reproduce any result.

In [ ]:
MLFLOW_AVAILABLE = False
try:
    import mlflow
    import mlflow.sklearn

    MLFLOW_AVAILABLE = True
    print(f"MLflow version: {mlflow.__version__}")
except ImportError:
    print("Install: pip install mlflow")

In [ ]:
if MLFLOW_AVAILABLE:
    mlflow.set_experiment("ds_study_classification")

    # Experiment 1: Logistic Regression
    with mlflow.start_run(run_name="logistic_regression"):
        # Log parameters
        mlflow.log_params({"model": "LogisticRegression", "C": 1.0, "max_iter": 1000})

        model1 = Pipeline(
            [("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))]
        )
        model1.fit(X_train, y_train)
        y_pred1 = model1.predict(X_test)

        acc1 = accuracy_score(y_test, y_pred1)
        f1_1 = f1_score(y_test, y_pred1)
        auc1 = roc_auc_score(y_test, model1.predict_proba(X_test)[:, 1])

        # Log metrics
        mlflow.log_metrics({"accuracy": acc1, "f1": f1_1, "roc_auc": auc1})

        # Log model artifact
        mlflow.sklearn.log_model(model1, artifact_path="model")
        print(f"LR  → acc={acc1:.4f}, f1={f1_1:.4f}, auc={auc1:.4f}")

    # Experiment 2: Random Forest
    with mlflow.start_run(run_name="random_forest"):
        mlflow.log_params(
            {"model": "RandomForest", "n_estimators": 100, "max_depth": 5}
        )

        model2 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        model2.fit(X_train, y_train)
        y_pred2 = model2.predict(X_test)

        acc2 = accuracy_score(y_test, y_pred2)
        f1_2 = f1_score(y_test, y_pred2)
        auc2 = roc_auc_score(y_test, model2.predict_proba(X_test)[:, 1])

        mlflow.log_metrics({"accuracy": acc2, "f1": f1_2, "roc_auc": auc2})
        mlflow.sklearn.log_model(model2, artifact_path="model")
        print(f"RF  → acc={acc2:.4f}, f1={f1_2:.4f}, auc={auc2:.4f}")

    print("\nView results: mlflow ui  (run in terminal, open http://localhost:5000)")
else:
    # Manual tracking without MLflow
    experiments = []

    for name, model in [
        (
            "LogisticRegression",
            Pipeline(
                [
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=1000)),
                ]
            ),
        ),
        (
            "RandomForest",
            RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
        ),
    ]:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        experiments.append(
            {
                "model": name,
                "accuracy": accuracy_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred),
            }
        )

    print(pd.DataFrame(experiments).to_string())
    print(
        "\nWith MLflow, all of this + hyperparameters + model artifacts would be auto-logged"
    )

---
## 2. Model Serialization — Saving and Loading

Before serving a model, you need to save (serialize) it to disk.

In [ ]:
# Train and save a model
model_final = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]
)
model_final.fit(X_train, y_train)

# Save with joblib (better than pickle for sklearn)
os.makedirs("saved_models", exist_ok=True)
joblib.dump(model_final, "saved_models/classifier_v1.joblib")
print("Model saved to: saved_models/classifier_v1.joblib")

# Also save metadata
metadata = {
    "model_name": "RandomForestClassifier",
    "version": "1.0",
    "feature_names": [f"feature_{i}" for i in range(X.shape[1])],
    "n_classes": 2,
    "test_accuracy": float(accuracy_score(y_test, model_final.predict(X_test))),
}
with open("saved_models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved. Test accuracy: {metadata['test_accuracy']:.4f}")

# Load and use
loaded_model = joblib.load("saved_models/classifier_v1.joblib")
sample = X_test[:3]
print(f"\nPredictions on 3 samples: {loaded_model.predict(sample)}")
print(f"Probabilities:")
print(loaded_model.predict_proba(sample).round(3))

---
## 3. FastAPI — Serving Your Model as a REST API

FastAPI lets you serve any model with a proper HTTP endpoint that other applications can call.

In [ ]:
# Write a FastAPI app to disk
api_code = """
# app.py — Run with: uvicorn app:app --reload

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
from typing import List

app = FastAPI(title="ML Model API", version="1.0")

# Load model at startup
model = joblib.load("saved_models/classifier_v1.joblib")


class PredictionRequest(BaseModel):
    features: List[float]   # list of 10 feature values


class PredictionResponse(BaseModel):
    prediction: int
    probability_class_1: float


@app.get("/")
def root():
    return {"message": "ML Model API is running"}


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    X = np.array(request.features).reshape(1, -1)
    prediction = int(model.predict(X)[0])
    probability = float(model.predict_proba(X)[0][1])
    return PredictionResponse(
        prediction=prediction,
        probability_class_1=round(probability, 4)
    )


@app.get("/health")
def health():
    return {"status": "healthy"}
"""

with open("app.py", "w") as f:
    f.write(api_code)

print("FastAPI app written to: app.py")
print()
print("To start the server:")
print("  pip install fastapi uvicorn")
print("  uvicorn app:app --reload")
print()
print("API will be available at:")
print("  GET  http://localhost:8000/         — root")
print("  POST http://localhost:8000/predict  — predictions")
print("  GET  http://localhost:8000/docs     — auto-generated Swagger UI")

In [ ]:
# Test the API locally without starting the server
# In production you'd call this with requests.post()

import json

# Simulated API call
test_payload = {
    "features": X_test[0].tolist()
}

# What the API would return
X_sample = np.array(test_payload['features']).reshape(1, -1)
prediction = int(loaded_model.predict(X_sample)[0])
probability = float(loaded_model.predict_proba(X_sample)[0][1])

response = {
    "prediction": prediction,
    "probability_class_1": round(probability, 4)
}

print("Simulated API request:")
print(f"  POST /predict")
print(f"  Body: {{features: [{', '.join([f'{x:.3f}' for x in X_test[0][:3]])}...]}}}")
print(f"\nResponse:")
print(json.dumps(response, indent=2))

---
## 4. Streamlit — Interactive ML Demos

Streamlit turns Python scripts into shareable web apps with zero frontend code.

In [ ]:
# Write a Streamlit app to disk
streamlit_code = """
# streamlit_app.py — Run with: streamlit run streamlit_app.py

import streamlit as st
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="ML Model Demo", layout="wide")

@st.cache_resource
def load_model():
    return joblib.load("saved_models/classifier_v1.joblib")

model = load_model()

st.title("Machine Learning Model Demo")
st.markdown("Adjust the sliders to input features and see predictions in real time.")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Input Features")
    features = []
    for i in range(10):
        val = st.slider(f"Feature {i+1}", min_value=-3.0, max_value=3.0, value=0.0, step=0.1)
        features.append(val)

with col2:
    st.subheader("Prediction")
    X_input = np.array(features).reshape(1, -1)
    prediction = model.predict(X_input)[0]
    probability = model.predict_proba(X_input)[0]

    st.metric("Predicted Class", prediction)
    st.metric("Confidence", f"{max(probability):.1%}")

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["Class 0", "Class 1"], probability, color=["steelblue", "coral"])
    ax.set_ylabel("Probability")
    ax.set_title("Class Probabilities")
    st.pyplot(fig)
"""

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("Streamlit app written to: streamlit_app.py")
print()
print("To run:")
print("  pip install streamlit")
print("  streamlit run streamlit_app.py")

---
## 5. Docker Concepts

Docker packages your app + all dependencies into a container that runs identically anywhere.

In [ ]:
dockerfile = """# Dockerfile for the FastAPI ML app

FROM python:3.11-slim

WORKDIR /app

# Copy requirements and install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app code and model
COPY app.py .
COPY saved_models/ ./saved_models/

# Expose port and start server
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

requirements = """fastapi
uvicorn[standard]
scikit-learn
joblib
numpy
pydantic
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Docker files created:")
print("  Dockerfile — container definition")
print("  requirements.txt — Python dependencies")
print()
print("Docker commands:")
print("  docker build -t ml-api .           # build the image")
print("  docker run -p 8000:8000 ml-api     # run the container")
print("  docker push username/ml-api        # push to Docker Hub")

---
## Summary — MLOps Stack

```
Development:
  Jupyter Notebooks → train and experiment
  MLflow             → track experiments, compare models, store artifacts

Deployment:
  joblib             → serialize trained model
  FastAPI            → REST API wrapping model.predict()
  Streamlit          → interactive demo for stakeholders
  Docker             → containerize everything for consistent deployment
```

**MLflow key commands:**
```bash
mlflow ui                           # open tracking UI at localhost:5000
mlflow models serve -m runs:/...    # serve any logged model
```

**Full deployment workflow:**
1. Train and track experiments with MLflow
2. Select best model, save with `joblib`
3. Wrap in FastAPI `predict` endpoint
4. Build Docker image
5. Deploy to cloud (AWS ECS, Google Cloud Run, Azure Container Apps)